# Colab HuggingFace Retrieval Experiment

이 노트북은 Colab에서 공유 데이터 기반 RAG 검색 실험을 실행하기 위한 실행 파일입니다.

목표:
- VM 디스크/CPU 부담을 줄이기 위해 Colab에서 index 생성과 prediction 생성을 실행합니다.
- OpenAI embedding 대신 HuggingFace embedding preset(`koe5`, `bge-m3`, `kure`)을 사용합니다.
- 기본 실험은 기존 최고 방식인 `Hybrid + Rerank`만 실행합니다.
- `VECTOR_STORE` 설정으로 `faiss` 또는 `chroma`를 선택할 수 있습니다.
- 결과 파일은 Google Drive에 저장해서 나중에 VM 결과와 합칩니다.

주의:
- 이 노트북에는 API key를 저장하지 않습니다.
- `--generate-answer`, `--multi-query`를 사용하지 않으면 OpenAI API key가 필요 없습니다.
- Colab은 VM의 `/home/codeit/shared_file/...` 경로를 볼 수 없습니다. 데이터는 Google Drive에 올려두고 Drive 경로로 읽어야 합니다.
- Chroma DB는 Drive에 직접 만들지 말고 `/content/chroma/...` 같은 Colab 로컬 경로에 만드세요.

In [ ]:
# 1. 실험 설정
# 아래 값만 본인 Drive 구조에 맞게 수정하면 됩니다.

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/data/rag_files')
PROJECT_DIR = Path('/content/chatbot')

# 프로젝트 코드를 Colab에 준비하는 방법 중 하나를 선택합니다.
# 1) GitHub repo를 쓸 경우 REPO_URL을 채웁니다.
# 2) Drive에 chatbot 폴더를 통째로 올려둔 경우 DRIVE_PROJECT_DIR을 사용합니다.
# 3) Drive에 chatbot.zip을 올려둔 경우 PROJECT_ZIP을 사용합니다.
REPO_URL = ''  # 예: 'https://github.com/your-team/chatbot.git'
DRIVE_PROJECT_DIR = DRIVE_ROOT / 'chatbot'
PROJECT_ZIP = DRIVE_ROOT / 'chatbot.zip'

# Drive에 올려둔 데이터 위치입니다.
DATA_PATHS = {
    'soyeon': DRIVE_ROOT / 'soyeon/chunks_v2_690.jsonl',
    'yongjun': DRIVE_ROOT / 'yongjun/rag_database.jsonl',
}

# eval csv가 repo에 없다면 Drive의 이 경로에서 PROJECT_DIR/data/eval로 복사합니다.
DRIVE_EVAL_DIR = DRIVE_ROOT / 'data/eval'

# 비용/시간 관리를 위해 기본은 두 데이터셋에 대해 Hybrid + Rerank만 실행합니다.
DATASETS_TO_RUN = ['soyeon', 'yongjun']
EMBEDDING_PRESET = 'koe5'  # 선택지: 'koe5', 'bge-m3', 'kure'
VECTOR_STORE = 'chroma'     # 선택지: 'faiss', 'chroma'
TOP_K = 5
RERANK = True
CANONICAL_ONLY = True

# Chroma를 쓸 때 collection 이름 prefix입니다.
CHROMA_COLLECTION_PREFIX = 'rfp'

# 이미 index/vector DB를 만들어둔 경우 False로 바꾸면 재생성을 건너뜁니다.
BUILD_INDEX = True
GENERATE_PREDICTIONS = True
RUN_EVALUATION = True

LOCAL_INDEX_ROOT = Path('/content/indexes')
LOCAL_CHROMA_ROOT = Path('/content/chroma')
DRIVE_OUTPUT_ROOT = DRIVE_ROOT / 'outputs/colab_hf_experiments'

assert VECTOR_STORE in {'faiss', 'chroma'}, "VECTOR_STORE는 'faiss' 또는 'chroma'여야 합니다."

print('DRIVE_ROOT:', DRIVE_ROOT)
print('PROJECT_DIR:', PROJECT_DIR)
print('DATASETS_TO_RUN:', DATASETS_TO_RUN)
print('EMBEDDING_PRESET:', EMBEDDING_PRESET)
print('VECTOR_STORE:', VECTOR_STORE)

In [ ]:
# 2. Google Drive mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab 모듈이 없습니다. 이 셀은 Colab 환경에서 실행하세요.')

In [ ]:
# 3. 프로젝트 코드 준비
# 우선순위: /content/chatbot 기존 폴더 -> Drive chatbot 폴더 복사 -> Drive zip 압축해제 -> Git clone

import os
import shutil
import subprocess
import sys
from pathlib import Path


def has_project_files(path: Path) -> bool:
    return (path / 'scripts/generate_eval_predictions.py').exists() and (path / 'src').exists()

if has_project_files(PROJECT_DIR):
    print('프로젝트가 이미 준비되어 있습니다:', PROJECT_DIR)
elif DRIVE_PROJECT_DIR.exists() and has_project_files(DRIVE_PROJECT_DIR):
    print('Drive 프로젝트 폴더를 /content로 복사합니다:', DRIVE_PROJECT_DIR)
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    ignore = shutil.ignore_patterns('.venv', '__pycache__', '.git', 'indexes', 'outputs')
    shutil.copytree(DRIVE_PROJECT_DIR, PROJECT_DIR, ignore=ignore)
elif PROJECT_ZIP.exists():
    print('Drive 프로젝트 zip을 압축 해제합니다:', PROJECT_ZIP)
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['unzip', '-q', str(PROJECT_ZIP), '-d', '/content'], check=True)
    if not has_project_files(PROJECT_DIR):
        candidates = [p for p in Path('/content').iterdir() if p.is_dir() and has_project_files(p)]
        if candidates:
            candidates[0].rename(PROJECT_DIR)
elif REPO_URL:
    print('GitHub repo를 clone합니다:', REPO_URL)
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    raise FileNotFoundError(
        '프로젝트 코드를 찾지 못했습니다. REPO_URL을 채우거나, Drive에 rag_project/chatbot 폴더 또는 chatbot.zip을 준비하세요.'
    )

assert has_project_files(PROJECT_DIR), f'프로젝트 파일이 올바르지 않습니다: {PROJECT_DIR}'
print('PROJECT_DIR ready:', PROJECT_DIR)

In [ ]:
# 4. 필요한 패키지 설치
# HuggingFace embedding + retrieval eval에 필요한 최소 패키지를 설치합니다.
# Chroma를 선택한 경우 chromadb도 함께 설치합니다.

import subprocess
import sys

packages = [
    'openai',
    'python-dotenv',
    'rank-bm25',
    'faiss-cpu',
    'sentence-transformers',
    'pandas',
    'numpy',
    'tqdm',
    'openpyxl',
]

if VECTOR_STORE == 'chroma':
    packages.append('chromadb')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('packages installed:', packages)

In [ ]:
# 5. 데이터와 eval 파일 확인
# Colab에서는 VM 공유 경로를 볼 수 없으므로, 데이터는 Drive에 있어야 합니다.

import shutil

missing = []
for dataset in DATASETS_TO_RUN:
    path = DATA_PATHS[dataset]
    print(dataset, '=>', path, 'exists=', path.exists())
    if not path.exists():
        missing.append(str(path))

if missing:
    raise FileNotFoundError('Drive에 데이터 파일이 없습니다:\n' + '\n'.join(missing))

project_eval_dir = PROJECT_DIR / 'data/eval'
if not (project_eval_dir / 'eval_batch_01.csv').exists():
    if DRIVE_EVAL_DIR.exists():
        print('Drive eval 폴더를 프로젝트로 복사합니다:', DRIVE_EVAL_DIR)
        project_eval_dir.parent.mkdir(parents=True, exist_ok=True)
        if project_eval_dir.exists():
            shutil.rmtree(project_eval_dir)
        shutil.copytree(DRIVE_EVAL_DIR, project_eval_dir)
    else:
        raise FileNotFoundError(
            f'eval CSV를 찾지 못했습니다. repo의 {project_eval_dir} 또는 Drive의 {DRIVE_EVAL_DIR}에 eval_batch_*.csv를 준비하세요.'
        )

print('eval_dir ready:', project_eval_dir)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_INDEX_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_CHROMA_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# 6. 실행 helper

import shlex
import subprocess
from datetime import datetime, timezone


def run_cmd(cmd, cwd=PROJECT_DIR):
    printable = ' '.join(shlex.quote(str(part)) for part in cmd)
    print('\n$', printable)
    subprocess.run([str(part) for part in cmd], cwd=str(cwd), check=True)


def chroma_collection_name(dataset: str) -> str:
    return f'{CHROMA_COLLECTION_PREFIX}_{dataset}_{EMBEDDING_PRESET}'.replace('-', '_')


def experiment_paths(dataset: str):
    chunks = DATA_PATHS[dataset]
    if VECTOR_STORE == 'chroma':
        index_dir = LOCAL_CHROMA_ROOT / f'chroma_{EMBEDDING_PRESET}_{dataset}'
    else:
        index_dir = LOCAL_INDEX_ROOT / f'faiss_{EMBEDDING_PRESET}_{dataset}'
    collection = chroma_collection_name(dataset)
    exp_name = f'hybrid_rerank_{dataset}_{EMBEDDING_PRESET}_{VECTOR_STORE}_canonical'
    pred_path = DRIVE_OUTPUT_ROOT / 'predictions' / f'{exp_name}.jsonl'
    eval_copy_dir = DRIVE_OUTPUT_ROOT / 'eval_outputs' / exp_name
    pred_path.parent.mkdir(parents=True, exist_ok=True)
    eval_copy_dir.parent.mkdir(parents=True, exist_ok=True)
    return chunks, index_dir, collection, exp_name, pred_path, eval_copy_dir


def vector_store_exists(index_dir: Path) -> bool:
    if VECTOR_STORE == 'chroma':
        return (index_dir / 'chroma.sqlite3').exists()
    return (index_dir / 'index.faiss').exists()


def add_vector_store_args(cmd: list, index_dir: Path, collection: str) -> list:
    cmd.extend([
        '--vector-store', VECTOR_STORE,
        '--index-dir', index_dir,
    ])
    if VECTOR_STORE == 'chroma':
        cmd.extend(['--chroma-collection', collection])
    return cmd

In [ ]:
# 7. Hybrid + Rerank 실험 실행
# 이 셀은 HuggingFace 모델 다운로드와 vector DB 생성 때문에 시간이 오래 걸릴 수 있습니다.
# GPU 런타임을 켜면 더 빠릅니다.

import shutil

completed = []

for dataset in DATASETS_TO_RUN:
    chunks, index_dir, collection, exp_name, pred_path, eval_copy_dir = experiment_paths(dataset)
    print('\n' + '=' * 100)
    print('DATASET:', dataset)
    print('VECTOR_STORE:', VECTOR_STORE)
    print('CHUNKS:', chunks)
    print('INDEX/DB:', index_dir)
    if VECTOR_STORE == 'chroma':
        print('CHROMA_COLLECTION:', collection)
    print('PREDICTION:', pred_path)
    print('EXPERIMENT:', exp_name)

    if BUILD_INDEX:
        if vector_store_exists(index_dir):
            print('vector store already exists, skip build:', index_dir)
        else:
            cmd = [
                sys.executable,
                'scripts/build_vector_index.py',
                '--chunks', chunks,
                '--embedding-preset', EMBEDDING_PRESET,
            ]
            add_vector_store_args(cmd, index_dir, collection)
            run_cmd(cmd)

    if GENERATE_PREDICTIONS:
        cmd = [
            sys.executable,
            'scripts/generate_eval_predictions.py',
            '--retriever', 'hybrid',
            '--chunks', chunks,
            '--embedding-preset', EMBEDDING_PRESET,
            '--top-k', str(TOP_K),
            '--canonical-only',
            '--output', pred_path,
            '--progress-every', '25',
        ]
        add_vector_store_args(cmd, index_dir, collection)
        if RERANK:
            cmd.append('--rerank')
        run_cmd(cmd)

    if RUN_EVALUATION:
        run_cmd([
            sys.executable,
            'eval/scripts/run_evaluation.py',
            '--predictions', pred_path,
            '--canonical-only',
            '--experiment-name', exp_name,
        ])
        source_eval_dir = PROJECT_DIR / 'eval/evaluation/outputs/eval'
        if eval_copy_dir.exists():
            shutil.rmtree(eval_copy_dir)
        shutil.copytree(source_eval_dir, eval_copy_dir)
        print('eval outputs copied to:', eval_copy_dir)

    completed.append(exp_name)

print('\ncompleted experiments:', completed)

In [ ]:
# 8. 결과 요약 확인
# eval log에서 방금 실행한 실험 행만 표시합니다.

import pandas as pd

log_path = PROJECT_DIR / 'eval/evaluation/outputs/eval/experiment_logs/phase1_retrieval_experiments.csv'
if log_path.exists():
    df = pd.read_csv(log_path)
    cols = [
        'experiment_name',
        'retriever_type',
        'embedding_model',
        'reranker',
        'overall_hit_at_5',
        'overall_mrr_at_5',
        'overall_ndcg_at_5',
        'num_eval_questions',
        'num_scored_questions',
    ]
    display(df[df['experiment_name'].isin(completed)][cols])
else:
    print('log file not found:', log_path)

print('Drive output root:', DRIVE_OUTPUT_ROOT)

## VM으로 결과 합치기

Colab 실행 후 Google Drive의 아래 폴더를 VM으로 가져와서 기존 결과와 비교하면 됩니다.

```text
/content/drive/MyDrive/rag_project/outputs/colab_hf_experiments/
```

주요 파일:
- `predictions/*.jsonl`
- `eval_outputs/<experiment_name>/eval_summary.md`
- `eval_outputs/<experiment_name>/eval_results.csv`
- `eval_outputs/<experiment_name>/experiment_logs/phase1_retrieval_experiments.csv`

로컬 vector store 위치:
- FAISS: `/content/indexes/...`
- Chroma: `/content/chroma/...`

Colab 런타임이 종료되면 로컬 vector store는 사라질 수 있습니다. 재사용하고 싶으면 Drive로 zip 저장하세요.

FAISS 예:

```bash
zip -r /content/drive/MyDrive/rag_project/indexes/faiss_koe5_soyeon.zip /content/indexes/faiss_koe5_soyeon
```

Chroma 예:

```bash
zip -r /content/drive/MyDrive/rag_project/indexes/chroma_koe5_soyeon.zip /content/chroma/chroma_koe5_soyeon
```